# Submission Demo — two-camera (fr+rl) localization

**Kernel:** `pixel-racers`  ·  **Backend:** `sp_nv_nn` (SuperPoint local + NetVLAD global + mutual-NN match)

输入为原始 `.mcap`。标定 / GT / 运动状态（`state_estimation`）都从 mcap 用 `tools/` 脚本现场提取并缓存。

### 整体数据流
```
原始 mcap
  ├─ tools/extract_assets.py → camera_info / tf_static / track_bounds / state_estimation.csv
  └─ tools/extract_gt.py     → gt.csv  (来自 /tf，仅 eval)
        │
  按圈切窗 (eval 用 GT 起终点穿越; predict 整段或手动窗口)
        │
  ┌───────────────┬────────────────────────┬──────────────────────────┐
  Option 1         Option 2                  Option 3
  motion only      two-camera localization   EKF fusion
  SE 航位积分       fr/rl 在线定位 → se2 融合   SE 预测 + fr/rl 外推 → se2 → EKF(R(s))
        │               │                           │
        └──── 每方案: submission.csv (timestamp_ns,x,y,yaw) ────┘
              eval 另出 metrics + tools/evaluate.py 打分
```

| Option | 相机特征 | 说明 |
|---|---|---|
| 1 motion only | **否** | 仅用 `state_estimation` 的 v/a/ω 航位积分，在两相机 t_last 采样 |
| 2 two-camera | 是 | fr+rl 在线 PnP 定位 → nearest-stamp ω 加权 SE(2) 融合（无 EKF）|
| 3 EKF fusion | 是 | SE 预测 + fr/rl 外推到 t_last → se2 融合 → EKF update，R(s)=σ(s)² |

每个 option cell：上方 markdown 详述算法流程，cell 内**显式写出流程**（调用库原语），跑完显示 official metrics 图 + 赛道轨迹图。
汇总 cell：三方案并排 metrics 柱状图 + 轨迹叠图。最后一个 cell：列出各 submission.csv 路径；eval 模式跑 `tools/evaluate.py` 打分。

`MODE="eval"`（training+GT，出 metrics） / `MODE="predict"`（test，仅 submission）。首跑在线 localize 慢；相同任务二跑读缓存。


## SETUP
载入配置、从 mcap 提取资源（camera_info/tf_static/track_bounds/state_estimation，eval 另提 gt）、载入标定与赛道线、解析运行窗口。
`ctx` 持有：资产、各相机标定、racing line、赛道边界、运行窗口、以及各 option 的结果缓存。

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # macOS libomp 冲突规避
os.environ.setdefault("OMP_NUM_THREADS", "1")

import sys
from pathlib import Path

HERE = Path.cwd()
if HERE.name != "submission-demo" and (HERE / "submission-demo").is_dir():
    HERE = HERE / "submission-demo"
for p in (str(HERE), str(HERE / "tools")):
    if p not in sys.path:
        sys.path.insert(0, p)

%matplotlib inline

# 重新加载 submission-demo 模块，使重跑 SETUP 能吸收代码改动（避免旧缓存导致 ImportError）
_local = {"pipeline", "plots", "sfm_core", "sfm_features", "sfm_localize",
          "fusion", "motion_ekf", "metrics", "obs_sigma", "mcap_io", "assets_io", "laps"}
for _m in list(sys.modules):
    if _m in _local or _m.split(".")[0] in _local:
        del sys.modules[_m]

import numpy as np
import pandas as pd
import pipeline as P
import motion_ekf as mek
import fusion, obs_sigma

# ---- paths (原始 mcap) ----
TRAIN_MCAP = "/Users/hbai/SchubKI/c0_2026-10-13_22-53-42_camloc/c0_2026-10-13_22-53-42_camloc_0.mcap"
TEST_MCAP  = None   # 填 test mcap 路径

# ---- run target ----
MODE = "eval"          # "eval" (training+GT, 出 metrics) | "predict" (test, 仅 submission)
LAPS = [3]             # None=全跑; 或 [3] / [5, 7]
WINDOWS   = None       # 可选 [(start_ns,end_ns),...]（predict 无 GT 切圈时）
LAPS_JSON = None       # 可选外部 lap 窗口 json

# ---- cameras / localize ----
CAMS         = ("fr", "rl")
REF_CAMERA   = "fr"
BACKEND      = "sp_nv_nn"
QUERY_STRIDE = 1       # 调大可快速冒烟（如 20）
TOP_K        = 5
OVERRIDE_QUERY = False

cfg = P.SubmissionConfig(
    train_mcap=TRAIN_MCAP, test_mcap=TEST_MCAP, mode=MODE,
    laps=LAPS, windows=WINDOWS, laps_json=LAPS_JSON,
    cams=CAMS, ref_camera=REF_CAMERA, backend=BACKEND,
    query_stride=QUERY_STRIDE, top_k=TOP_K, override_query=OVERRIDE_QUERY,
)
ctx = P.ensure_assets(cfg)
print("run dir:", cfg.run_dir)
print("windows:", [w["name"] for w in ctx.windows])

## Option 1 — motion prediction only

**纯航位推算，不触碰相机图像/特征。** 只用车端 `state_estimation` 的 body 系速度/加速度/偏航角速度积分。

**流程**
1. `window_se` — 把 `state_estimation` 切到当前圈窗口，得到 `vx,vy,ax,ay,wz` 时序。
2. `init_pose` — 首帧初值：eval 取 GT 首帧 pose；predict 取 SE 首帧 pose。
3. `camera_tlast` — **只读相机 header 时间戳**（不解码 JPEG、不提特征），用 fr/rl 最近邻同步得到每个逻辑帧的 `t_last = max(t_fr, t_rl)`。这是为了让三方案落在同一条时间轴上。
4. `dead_reckon_se` — 从初值积分 `Δp = v·Δt + ½a·Δt²`、`ψ += ω·Δt`（body→map 用区间中点航向），再插值采样到各 `t_last`。
5. `finalize` — 写 `submission.csv`；eval 算 official metrics（成功率 100%，因为每个 t_last 都输出一个预测位姿）并画图。


In [ ]:
for w in ctx.windows:
    se_win = P.window_se(ctx, w)                         # 1. SE 切窗
    x0, y0, yaw0, t0 = P.init_pose(ctx, se_win)          # 2. 首帧初值
    t_last = P.camera_tlast(ctx, w)                      # 3. 相机时间戳（无特征）
    pred = mek.run_mode_a_se_predict_at(                 # 4. 航位积分 + 采样
        se_win, t_last, x0=x0, y0=y0, yaw0=yaw0)
    out_dir = cfg.run_dir / w["name"] / "motion_only"
    P.finalize(ctx, pred, out_dir, "motion_only", w,     # 5. 落盘 + metrics + 图
               n_total=len(pred), show_plots=True)

## Option 2 — two-camera localization (fr + rl)

**纯视觉，两相机离线融合，无 EKF、无时间外推。**

**流程**
1. `ensure_loc_tables` — 对 fr、rl 各跑在线 SfM 定位（首跑）或读缓存（二跑）：对每帧做 NetVLAD top-K 检索 → mutual-NN 匹配 → 2D-3D → PnP RANSAC，得到每相机 `errors.csv`（`stamp_ns, ok, pred_x/y/yaw, n_pnp_inl` 等）。
2. `fuse_loc_tables_se2` — 以 `REF_CAMERA` 为锚做最近邻时间对齐，按置信度 `ω = n_inl·exp(-e_reproj/σ)` 做 SE(2) 加权融合，带位姿一致性门控（Δxy≤5m, Δyaw≤15°）。
3. 只输出融合成功（`ok`）的帧 → `submission.csv`。**成功率 = 融合成功帧 / 全部同步帧**（视觉失败的帧被丢弃，会拉低成功率/coverage）。
4. `finalize` — 写 submission + `errors_se2.csv`；eval 算 metrics 并画图。


In [ ]:
for w in ctx.windows:
    tables = P.ensure_loc_tables(ctx, w)                 # 1. fr/rl 在线定位(或缓存)
    fused = fusion.fuse_loc_tables_se2(                  # 2. 最近邻对齐 + ω 加权 se2 融合
        tables, ref_camera=ctx.cfg.ref_camera)
    ok = fused[fused.ok].reset_index(drop=True)          # 3. 只保留融合成功帧
    pred = pd.DataFrame({
        "timestamp_ns": ok.stamp_ns.to_numpy(np.int64),
        "x": ok.pred_x.to_numpy(float),
        "y": ok.pred_y.to_numpy(float),
        "yaw": ok.pred_yaw.to_numpy(float),
    })
    out_dir = cfg.run_dir / w["name"] / "two_cam"
    out_dir.mkdir(parents=True, exist_ok=True)
    fused.to_csv(out_dir / "errors_se2.csv", index=False)
    P.finalize(ctx, pred, out_dir, "two_cam", w,         # 4. 落盘 + metrics(分母=全部同步帧) + 图
               n_total=len(fused), show_plots=True)

## Option 3 — EKF fusion (fr + rl)

**运动预测 + 视觉校正的 SE(2) EKF**，把异步相机时序与随位置变化的观测不确定性写进统一滤波。

**流程**
1. `write_process_noise_yaml` + `NoiseCfg` — 从 `error-model/se_step_residual_noise.json` 的 MADσ 生成过程噪声 Q。
2. `ensure_loc_tables` — 同 Option 2 的 fr/rl 定位结果（复用缓存）。
3. `build_sync_buckets` — fr/rl 最近邻同步成逻辑帧，`t_last = max 时间戳`。
4. `run_mode_b_ekf` — 对每个 bucket：
   - **predict**：用 SE 的 v/a/ω 把状态从上一步推进到 `t_last`，协方差 `P = F P Fᵀ + Q`；
   - **外推**：每相机 PnP 位姿用同一航位模型从各自 stamp 外推到 `t_last`；
   - **融合**：`fuse_se2_weighted` 得单次观测 z；
   - **update**：`R(s)=diag(σ(s)², σ(s)², σ_yaw²)`（σ(s) 来自 `error-model/obs_error_sigma_vs_s.json`），标准 EKF 更新。视觉缺失则只预测。
5. `finalize` — 写 submission（每个 t_last 都有预测位姿，成功率 100%）+ `sim/ekf_sim.csv`（每帧 ok/权重/σ/P 对角）；eval 算 metrics 并画图。


In [ ]:
for w in ctx.windows:
    se_win = P.window_se(ctx, w)
    noise = mek.write_process_noise_yaml(Path(cfg.noise_json))   # 1. 过程噪声 Q
    ncfg = mek.NoiseCfg.from_dict(noise)
    x0, y0, yaw0, t0 = P.init_pose(ctx, se_win)

    tables = P.ensure_loc_tables(ctx, w)                         # 2. fr/rl 定位(缓存)
    buckets = mek.build_sync_buckets(                            # 3. fr/rl 同步成逻辑帧
        tables, ref_camera=cfg.ref_camera, cams=cfg.cams)
    pred, sim = mek.run_mode_b_ekf(                              # 4. 预测 → 外推 → se2 → update(R(s))
        se_win, buckets, ncfg,
        x0=x0, y0=y0, yaw0=yaw0, t0_ns=t0,
        sigma_of_s_fn=obs_sigma.sigma_of_s,
        track_line=ctx.track_line, cams=cfg.cams,
    )
    out_dir = cfg.run_dir / w["name"] / "ekf"
    (out_dir / "sim").mkdir(parents=True, exist_ok=True)
    sim.to_csv(out_dir / "sim" / "ekf_sim.csv", index=False)
    P.finalize(ctx, pred, out_dir, "ekf", w,                    # 5. 落盘 + metrics + 图
               n_total=len(pred), show_plots=True)

## 汇总对比
`run_summary` 对三方案做并排 official metrics 柱状图（pos RMSE xy / s/n、yaw RMSE、within 1m）与三条轨迹叠在赛道平面的对比图；eval 另写 `compare/metrics_compare.csv`。predict 模式只画轨迹叠图并列出 submission。

In [ ]:
mdf = P.run_summary(ctx, show_plots=True)
mdf

## 最终交付：submission.csv 路径 + 官方评分
`list_outputs` 列出每个 option 的 `submission.csv` 路径；**eval 模式**对每份 submission 跑 `tools/evaluate.py`（官方打分：position RMSE 为 SCORE、coverage ≥90% 才计入排名），把报告打印出来供 reviewer 核对。

In [ ]:
deliverables = P.list_outputs(ctx, evaluate=ctx.assets.has_gt)
deliverables